## Імпорт необхідних бібліотек

In [14]:
import urllib.request
import zipfile
import os
import pandas as pd
import numpy as np
import timeit
from datetime import time

url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
zip_path = "household_power_consumption.zip"
extract_folder = "power_data"

if not os.path.exists(extract_folder):
    os.makedirs(extract_folder)
    print("Завантаження датасету...")
    urllib.request.urlretrieve(url, zip_path)

    # Розпакування архіву
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
    print("Розпакування завершено")
else:
    print("Датасет вже завантажено та розпаковано")

Датасет вже завантажено та розпаковано


## Зчитування даних та Data Cleaning

In [4]:
file_path = os.path.join(extract_folder, "household_power_consumption.txt")
df_power = pd.read_csv(file_path, sep=';', na_values=['?'], low_memory = False)

df_clean = df_power.dropna().reset_index(drop=True)
print(f"Дані очищено. Розмір датасету: {df_clean.shape[0]} рядків.")
display(df_clean.head())

Дані очищено. Розмір датасету: 2049280 рядків.


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт

In [5]:
def filter_power(df):

    df['Global_active_power'] = pd.to_numeric(df['Global_active_power'], errors='coerce')

    result = df[df['Global_active_power'] > 5]
    
    return result

def new_func():
    filter_power = pd.to_numeric()

high_power_df = filter_power(df_clean)
print(f"Знайдено записів: {len(high_power_df)}")
display(high_power_df.head())

execution_time = timeit.timeit(stmt="filter_power(df_clean)", globals=globals(), number=1)
print(f"Час виконання: {execution_time:.4f} секунд")

Знайдено записів: 17547


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


Час виконання: 0.0065 секунд


### Записи, де сила струму 19-20 А, і пралка/холодильник споживають більше за бойлер/кондиціонер

In [7]:
def current_strength(df):
    df_copy = df.copy()

    convert = ['Global_intensity', 'Sub_metering_2', 'Sub_metering_3']
    for col in convert:
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')

        result = df_copy[
            (df_copy['Global_intensity'] >= 19) &
            (df_copy['Global_intensity'] <= 20) &
            (df_copy['Sub_metering_2'] > df_copy['Sub_metering_3'])
        ]

        return result
appliances_df = current_strength(df_clean)
print(f"Записів знайдено: {len(appliances_df)}")
display(appliances_df.head())

execution_time = timeit.timeit(stmt="current_strength(df_clean)", globals=globals(), number=1)
print(f"Час виконання: {execution_time:.4f} секунди")

Записів знайдено: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


Час виконання: 0.0861 секунди


### Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії

In [12]:
def get_random_means(df):
    df_copy = df.copy()
    cols_to_convert = ['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    for col in cols_to_convert:
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')

    random_df = df_copy.sample(n=500000)
    result = random_df[cols_to_convert].mean()
    return result

means_result = get_random_means(df_clean)
print("Середні значення для 500 000 випадкових записів:")
print(means_result)

execution_time = timeit.timeit(stmt="get_random_means(df_clean)", globals=globals(), number=1)
print(f"Час виконання: {execution_time:.4f} секунди")

Середні значення для 500 000 випадкових записів:
Sub_metering_1    1.119030
Sub_metering_2    1.286840
Sub_metering_3    6.467532
dtype: float64
Час виконання: 0.1741 секунди


### Обрати ті записи, які після 18-00 споживають понад 6 кВт за хвилину в середньому, серед відібраних визначити ті, у яких основне споживання електроенергії у вказаний проміжок часу припадає на пральну машину, сушарку, холодильник та освітлення (група 2 є найбільшою), а потім обрати кожен третій результат із першої половини та кожен четвертий результат із другої половини.


In [18]:
def complex_filter(df):
    df_copy = df.copy()
    
    df_copy['Time'] = pd.to_datetime(df_copy['Time'].astype(str), format = '%H:%M:%S', errors='coerce').dt.time

    cols = ['Global_active_power', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
    for col in cols:
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')

    df_step1 = df_copy[
        (df_copy['Time'] > time(18, 0)) &
        (df_copy['Global_active_power'] > 6)
    ]

    df_step2 = df_step1[
        (df_step1['Sub_metering_2'] > df_step1['Sub_metering_1']) &
        (df_step1['Sub_metering_2'] > df_step1['Sub_metering_3'])
    ]

    half_index = len(df_step2) // 2
    part1 = df_step2.iloc[:half_index:3]
    part2 = df_step2.iloc[half_index::4]
    result = pd.concat([part1, part2])
    return result

final_df = complex_filter(df_clean)
print(f"Фінальна кількість рядків: {len(final_df)}")
display(final_df.head())

execution_time = timeit.timeit(stmt="complex_filter(df_clean)", globals=globals(), number=1)
print(f"Час виконання: {execution_time:.4f} секунди")

Фінальна кількість рядків: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17492,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17496,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17499,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


Час виконання: 2.9780 секунди


### Пронормувати та стандартизувати вибраний датасет


In [21]:
df_numeric = df_clean[['Global_active_power', 'Global_intensity', 'Sub_metering_1']].dropna().sample(5000)

df_normalized = (df_numeric - df_numeric.min()) / (df_numeric.max() - df_numeric.min())
df_standardized = (df_numeric - df_numeric.mean()) / df_numeric.std()
print("Нормовані дані: ")
display(df_normalized.head())
print("Стандартизовані дані: ")
display(df_standardized.head())


Нормовані дані: 


,Global_active_power,Global_intensity,Sub_metering_1
821335,0.017881,0.024876,0.0
1044515,0.035107,0.039801,0.0
1128495,0.026821,0.029851,0.0
1737209,0.141300,0.134328,0.0
1612502,0.137375,0.134328,0.0


Стандартизовані дані: 


,Global_active_power,Global_intensity,Sub_metering_1
821335,-0.781944,-0.750532,-0.183327
1044515,-0.634729,-0.617590,-0.183327
1128495,-0.705541,-0.706218,-0.183327
1737209,0.272787,0.224381,-0.183327
1612502,0.239245,0.224381,-0.183327


### Підрахувати коефіцієнт Пірсона та Спірмена для двох integer/real атрибутів.

In [23]:
df_numeric = df_clean[['Global_active_power', 'Global_intensity', 'Sub_metering_1']].dropna().sample(5000)

pearson = df_numeric['Global_active_power'].corr(df_numeric['Global_intensity'], method='pearson')
spearman = df_numeric['Global_active_power'].corr(df_numeric['Global_intensity'], method='spearman')

print(f"\nКореляція Пірсона: {pearson:.4f}")
print(f"Кореляція Спірмена: {spearman:.4f}")


Кореляція Пірсона: 0.9990
Кореляція Спірмена: 0.9951


### Провести One Hot Encoding категоріального атрибута.

In [25]:
df_clean['Date'] = pd.to_datetime(df_clean['Date'], format='%d/%m/%Y', errors='coerce')
df_clean['Weekday'] = df_clean['Date'].dt.day_name()

df_encoded = pd.get_dummies(df_clean, columns=['Weekday'], dtype=int)
print("\nДані після One Hot Encoding:")
display(df_encoded.iloc[:, -7:].head())


Дані після One Hot Encoding:


,Weekday_Friday,Weekday_Monday,Weekday_Saturday,Weekday_Sunday,Weekday_Thursday,Weekday_Tuesday,Weekday_Wednesday
0,0,0,1,0,0,0,0
1,0,0,1,0,0,0,0
2,0,0,1,0,0,0,0
3,0,0,1,0,0,0,0
4,0,0,1,0,0,0,0
